## import

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
# import re
# from collections import Counter
# import random
from tqdm import tqdm

import numpy as np

# from time import sleep
# from time import time
# from random import randint

import logging
from pathlib import Path


In [ ]:
def format_fpth(fpth):
    if isinstance(fpth, Path):
        return fpth.as_posix()
    if isinstance(fpth, str):
        return fpth
    raise FileNotFoundError
def dump_json(obj, fpth, mode="w", encoding="utf-8", indent=4):
    import codecs
    import json
    with codecs.open(format_fpth(fpth), encoding=encoding, mode=mode) as fw:
        json.dump(obj, fw, ensure_ascii=False, indent=indent)


## bert run

* bert-chinese, google-bert/bert-base-multilingual-uncased, google-bert/bert-base-multilingual-uncased
* roberta-chinese, FacebookAI/xlm-roberta-large
* xlnet,
* bart,
* simsce,
* ernie,
* t5,

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

In [ ]:
fin = "/content/drive/MyDrive/mwe/test-pairs-ids.json"
df = pd.read_json(fin, lines=True)

for model_name in ["google-bert/bert-base-chinese", "google-bert/bert-base-multilingual-uncased",
                   "google-bert/bert-base-multilingual-cased",   # bert
                   "hfl/chinese-bert-wwm", # bert-wmm
                   "hfl/chinese-roberta-wwm-ext", # roberta-wmm
                   "FacebookAI/xlm-roberta-large", # roberta没有中文版本
                   "hfl/chinese-xlnet-base"
                   ]:

    fout = "/content/drive/MyDrive/mwe/test-{}.json".format(model_name.split("/")[-1])
    if Path(fout).exists():
        continue

    print(">> ", model_name)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.to("cuda")

    logging.info("** sample count: {}".format(df.shape[0]))
    logging.info("** model: {}".format(model_name))

    pbar = tqdm(total=len(df))
    scores = {}
    for _, row in df.iterrows():

        _id, sentences = row["id"], row["pair"]


        encoded_input = tokenizer(sentences, return_tensors='pt',
                                  padding=True, max_length=512, truncation=True)
        encoded_input.to("cuda")

        with torch.no_grad():

            sentence_embeddings = model(**encoded_input).last_hidden_state
            sentence_embeddings = sentence_embeddings.mean(dim=1).cpu().detach().numpy()

        scr = 1 - paired_cosine_distances([sentence_embeddings[0]],[sentence_embeddings[1]])
        scores[_id] = str(np.round(scr[0], 6))

        pbar.update(1)
    pbar.close()

    dump_json(scores, fout)

## sbert run

In [ ]:

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import paired_cosine_distances

In [ ]:
def sbert_infer(sentences,  model):
    # model = SentenceTransformer(model)
    # sentences = ['那个人很开心', '那个人非常开心']
    sentence_embeddings = model.encode(sentences)

    score = 1 - paired_cosine_distances([sentence_embeddings[0]],[sentence_embeddings[1]])

    return score

In [ ]:

import sys
from pathlib import Path
sys.path.insert(1, Path.cwd().as_posix())
sys.path.insert(1, Path.cwd().parent.as_posix())

import logging
# from common.io_utils import *
# from common.setup import logging, adr


fin = "/content/drive/MyDrive/mwe/test-pairs-newID.json"
df = pd.read_json(fin, lines=True)

for model_name in ["uer/sbert-base-chinese-nli",   ## ！！ 有大大的问题
                   "DMetaSoul/sbert-chinese-general-v2",
                  #  "paraphrase-multilingual-MiniLM-L12-v2",
                   "paraphrase-multilingual-mpnet-base-v2",
                   "distiluse-base-multilingual-cased-v1"]:

    fout = "/content/drive/MyDrive/mwe/test-{}.json".format(model_name.split("/")[-1])

    model = SentenceTransformer(model_name)
    logging.info("** sample count: {}".format(df.shape[0]))
    logging.info("** model: {}".format(model_name))

    pbar = tqdm(total=len(df))
    scores = {}
    for _, row in df.iterrows():

        _id, pair = row["id"], row["pair"]

        sentences = pair
        sentence_embeddings = model.encode(sentences)

        scr = 1 - paired_cosine_distances([sentence_embeddings[0]],[sentence_embeddings[1]])
        scores[_id] = str(np.round(scr[0], 6))

        pbar.update(1)
    pbar.close()

    dump_json(scores, fout)

## hellonlp

In [ ]:
!pip install promcse

In [ ]:
from promcse import PromCSE
import torch
from transformers import BertTokenizer
from transformers import BertModel
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
def get_vector_simcse(sentence):
    """
    预测simcse的语义向量。
    """
    input_ids = torch.tensor(tokenizer.encode(sentence)).unsqueeze(0).to("cuda")
    output = model(input_ids)
    return output.last_hidden_state[:, 0].squeeze(0)

def get_similarity_two(sentence1, sentence2):
    vec1 = get_vector_simcse(sentence1).tolist()
    vec2 = get_vector_simcse(sentence2).tolist()
    similarity_list = cosine_similarity([vec1], [vec2]).tolist()[0][0]
    return similarity_list


In [ ]:

import sys
from pathlib import Path
sys.path.insert(1, Path.cwd().as_posix())
sys.path.insert(1, Path.cwd().parent.as_posix())

import logging
# from common.io_utils import *
# from common.setup import logging, adr


fin = "/content/drive/MyDrive/mwe/test-pairs-newID.json"
df = pd.read_json(fin, lines=True)

for model_name in ["hellonlp/promcse-bert-base-zh",
                   "hellonlp/promcse-bert-base-zh-v1.1",
                   "hellonlp/promcse-bert-large-zh",
                   "hellonlp/simcse-roberta-base-zh",
                   "hellonlp/simcse-roberta-large-zh"
                   ]:

    fout = "/content/drive/MyDrive/mwe/test-{}.json".format(model_name.split("/")[-1])
    if Path(fout).exists():
      continue

    if "promcse" in model_name:
      model = PromCSE(model_name, "cls", 10)
    else:
      model = BertModel.from_pretrained(model_name)
      tokenizer = BertTokenizer.from_pretrained(model_name)
      model.to("cuda")


    # model = SentenceTransformer(model_name)
    logging.info("** sample count: {}".format(df.shape[0]))
    logging.info("** model: {}".format(model_name))

    pbar = tqdm(total=len(df))
    scores = {}
    for _, row in df.iterrows():

        _id, [s1, s2] = row["id"], row["pair"]
        if "promcse" in model_name:
          scr = model.similarity(s1, s2)
        else:
          scr = get_similarity_two(s1, s2)

        scores[_id] = str(np.round(scr, 6))

        pbar.update(1)
    pbar.close()

    dump_json(scores, fout)

In [ ]:
!pip install promcse

In [ ]:
from promcse import PromCSE


fin = "/content/drive/MyDrive/mwe/test-pairs-newID.json"
df = pd.read_json(fin, lines=True)

for model_name in ["YuxinJiang/unsup-promcse-bert-base-uncased",
                   "YuxinJiang/sup-promcse-roberta-base",
                   "YuxinJiang/sup-promcse-roberta-large",
                   ]:

    fout = "/content/drive/MyDrive/mwe/test-{}.json".format(model_name.split("/")[-1])
    if Path(fout).exists():
      continue

    try:
       model = PromCSE(model_name, "cls_before_pooler", 16)
    except:
       model = PromCSE(model_name, "cls_before_pooler", 10)
       
    logging.info("** sample count: {}".format(df.shape[0]))
    logging.info("** model: {}".format(model_name))

    pbar = tqdm(total=len(df))
    scores = {}
    for _, row in df.iterrows():

        _id, [s1, s2] = row["id"], row["pair"]
        if "promcse" in model_name:
          scr = model.similarity(s1, s2)
        else:
          scr = get_similarity_two(s1, s2)

        scores[_id] = str(np.round(scr, 6))

        pbar.update(1)
    pbar.close()

    dump_json(scores, fout)

# scorer

In [ ]:
import csv
from scipy.stats import pearsonr, spearmanr

def _score( pred, gt):
    """
    pred: dataframe or dict?
    gt: pandas dataframe
    """

    gold_labels_all = list()
    predictions_all = list()

    gold_labels_sts = list()
    predictions_sts = list()

    gold_labels_no_sts = list()
    predictions_no_sts = list()

    for _, row in gt.iterrows():
        this_sim = row["sim"]
        id1 = row["ID"]
        try:
            if pd.isna(this_sim):
                id2 = row["otherID"]
                this_sim = pred[id2]

            this_sim = float( this_sim )
            this_prediction = round(float( pred[id1] ), 2)
        except Exception as err:
            print("******", err)
            break

        if pd.isna(row["idiom"]):  # standard sts
            gold_labels_sts.append( this_sim        )
            predictions_sts.append( this_prediction )
        else :
            gold_labels_no_sts.append( this_sim        )
            predictions_no_sts.append( this_prediction )

        gold_labels_all.append( this_sim        )
        predictions_all.append( this_prediction )

    corel_all, pvalue    =  spearmanr(gold_labels_all   , predictions_all    )
    corel_sts, pvalue    =  spearmanr(gold_labels_sts   , predictions_sts    )
    corel_no_sts, pvalue =  spearmanr(gold_labels_no_sts, predictions_no_sts )
    return ( corel_all, corel_sts, corel_no_sts )

In [ ]:
def load_json(fpth, mode="r", encoding="utf-8"):
    import json

    with open(format_fpth(fpth), mode=mode, encoding=encoding) as fr:
        data = json.load(fr)

    return data

In [ ]:
from pathlib import Path

_dir = Path("")  # directory to prediction files
gpth = ""  # path to gt file

for fpth in _dir.glob("test-*.json"):
    if "test-pairs" in fpth.stem:
      continue
    print(">> ", fpth.stem.split("test-", 1)[1])
    pred = load_json(fpth)


    gt = pd.read_json(gpth, lines=True)
    gt = gt[gt["set"]=="test"]


    print("sample count: ", len(pred), len(gt))
    corel_all, corel_sts, corel_no_sts = _score( pred, gt)
    print(corel_all, corel_no_sts, corel_sts)
